In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df=pd.read_csv('H2.csv')

df["Company"]=df["Company"].transform(lambda x: np.nan if x == "       NULL" else x)
df["Agent"]=df["Agent"].transform(lambda x: np.nan if x == "       NULL" else x)

In [ ]:
#EDA ploting
missing=df.isna().sum()
missing=missing[missing>0]
missing
plt.bar(missing.index,missing.values,color="#D1495B")
plt.title("Missing values")
plt.xlabel("Missing Columns")
plt.ylabel("Missing Values")

In [ ]:
plt.bar(["Not cancelled", "Cancelled"],
        df["IsCanceled"].value_counts().values,color="#D1495B")
plt.title("Cancelled vs. Not Cancelled")

In [ ]:
import seaborn as sns
num = ["LeadTime", "StaysInWeekendNights", "StaysInWeekNights", "Adults",
        "Children", "Babies", "PreviousCancellations", "PreviousBookingsNotCanceled",
        "BookingChanges", "DaysInWaitingList", "ADR", "RequiredCarParkingSpaces",
        "TotalOfSpecialRequests"]
fig, axes = plt.subplots(4, 4, figsize=(18, 13))
for ax, c in zip(axes.ravel(), num):
    sns.histplot(df[c].dropna(), bins=40, ax=ax, color="#2E86AB")
    ax.set_yscale("log"); ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[len(num):]:
    ax.axis("off")
fig.suptitle("Numeric feature distributions (log-scaled counts)", y=1.01, fontsize=14)

In [ ]:
cats = ["Meal", "MarketSegment", "DistributionChannel", "DepositType",
            "CustomerType", "ReservedRoomType", "IsRepeatedGuest", "ArrivalDateYear"]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, c in zip(axes.ravel(), cats):
    r = df.groupby(c)["IsCanceled"].agg(["mean", "size"]).sort_values("mean")
    r = r[r["size"] >= 30]                       # skip tiny groups
    ax.barh(r.index.astype(str), r["mean"] * 100, color="#D1495B")
    ax.axvline(df["IsCanceled"].mean() * 100, color="k", ls="--", lw=1)
    ax.set_title(f"Cancellation % by {c}"); ax.set_xlabel("%")
fig.suptitle("Cancellation rate by category (dashed = overall rate)", y=1.01, fontsize=14)

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

df["Month"] = pd.Categorical(df["ArrivalDateMonth"], MONTHS, ordered=True)
cr = df.groupby("Month", observed=True)['IsCanceled'].mean() * 100
plt.plot(cr.index, cr.values, marker="o", color="#D1495B")
plt.title("Cancellation % per arrival month")
plt.grid(axis="both")
plt.tick_params(axis="x", rotation=60)

In [ ]:
adr = df.groupby(["Month", "ArrivalDateYear"], observed=True)["ADR"].median().unstack()
plt.plot(adr.index,adr.values,marker='o')
plt.grid(axis="both")
plt.title("Median ADR per arrival month")
plt.tick_params(axis='x', rotation=60)

In [ ]:
corr = df.select_dtypes("number").drop(columns=["Agent", "Company"], errors="ignore").corr()
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7}, ax=ax, cbar_kws={"shrink": .8})
ax.set_title("Correlation matrix (numeric features)")